# LeetCode 3501 — Maximize Active Section with Trade II bằng Python

- Link: https://leetcode.com/problems/maximize-active-section-with-trade-ii/
- Độ khó: **Hard**
- Chủ đề: **Nén chuỗi thành các khối** (run-length), **tổng hai khối 0 liên tiếp**, **RMQ bằng sparse table**, xử lý **khối bị cắt** ở biên truy vấn bằng `bisect`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxActiveSectionsAfterTrade(self, s: str, queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Hiểu "giao dịch": biến một khối `1` **bị kẹp giữa hai khối `0`** thành `0` (ba khối gộp thành một khối `0` lớn), rồi biến một khối `0` bị kẹp giữa các `1` thành `1` ⇒ lợi ích tốt nhất = **tổng độ dài hai khối `0` kề khối `1` đó**.
2. Đếm kết quả trên **toàn chuỗi `s`**: số `1` ban đầu của `s` cộng lợi ích lớn nhất trong đoạn truy vấn.
3. Liệt kê các **khối 0** của `s` (đầu, cuối); một truy vấn `[l, r]` cắt ra dãy khối 0 liên tiếp, hai khối ở biên có thể **bị cắt ngắn**.
4. Trả lời `max` tổng cặp khối 0 liên tiếp trên đoạn bằng **sparse table** `O(1)`, xử lý riêng hai cặp ở biên.
5. Kiểm chứng bằng mô phỏng mọi giao dịch trên chuỗi tăng cường `'1' + sub + '1'`.

## 2. Đề bài & ví dụ

Chuỗi nhị phân `s` (`'1'` = hoạt động). Một **giao dịch** (tối đa một lần):

1. Đổi một khối `1` liên tiếp **được bao bởi `0` hai bên** thành toàn `0`;
2. Sau đó đổi một khối `0` liên tiếp **được bao bởi `1` hai bên** thành toàn `1`.

Với mỗi truy vấn `[l, r]`, xét `t = '1' + s[l..r] + '1'` (hai số `1` thêm vào không được đếm), thực hiện giao dịch tối ưu **chỉ trên đoạn** đó, và trả về **số `1` lớn nhất của cả chuỗi `s`** sau giao dịch. Các truy vấn độc lập.

**Ví dụ 2:** `s = "0100"`, truy vấn `[[0,3],[0,2],[1,3],[2,3]]` → `[4,3,1,1]`.

**Ví dụ 3:** `s = "1000100"`, `[[1,5],[0,6],[0,4]]` → `[6,7,2]`.

**Ví dụ 4:** `s = "01010"`, `[[0,3],[1,4],[1,3]]` → `[4,4,2]`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ q ≤ 10^5`.

In [1]:
from typing import List
from bisect import bisect_left, bisect_right
examples = [
    ("01", [[0, 1]], [1]),
    ("0100", [[0, 3], [0, 2], [1, 3], [2, 3]], [4, 3, 1, 1]),
    ("1000100", [[1, 5], [0, 6], [0, 4]], [6, 7, 2]),
    ("01010", [[0, 3], [1, 4], [1, 3]], [4, 4, 2]),
]
for s, q, exp in examples:
    print(repr(s), q, '-> kỳ vọng', exp)

'01' [[0, 1]] -> kỳ vọng [1]
'0100' [[0, 3], [0, 2], [1, 3], [2, 3]] -> kỳ vọng [4, 3, 1, 1]
'1000100' [[1, 5], [0, 6], [0, 4]] -> kỳ vọng [6, 7, 2]
'01010' [[0, 3], [1, 4], [1, 3]] -> kỳ vọng [4, 4, 2]


## 3. Kiến thức nền

### 3.1. Nén thành khối (run-length)

In [2]:
def runs(t):
    out, i = [], 0
    while i < len(t):
        j = i
        while j < len(t) and t[j] == t[i]:
            j += 1
        out.append((t[i], i, j - 1))
        i = j
    return out

print(runs("1000101"))

[('1', 0, 0), ('0', 1, 3), ('1', 4, 4), ('0', 5, 5), ('1', 6, 6)]


### 3.2. Lợi ích của một giao dịch

Chọn khối `1` độ dài `O` kẹp giữa khối `0` trái `Z_L` và phải `Z_R`. Bước 1 tạo khối `0` dài `Z_L + O + Z_R`, bước 2 đổi nó thành `1` (nó được bao bởi `1` hai bên vì `Z_L`, `Z_R` vốn bị kẹp bởi `1` hoặc các số `1` tăng cường). Số `1` thay đổi: `−O + (Z_L + O + Z_R) = Z_L + Z_R`. Đổi một khối `0` **khác** ở bước 2 chỉ được ≤ một khối `0` ban đầu ⇒ không tốt hơn.

In [3]:
t = "1" + "00010" + "1"
print('khối:', runs(t), '| giao dịch quanh khối 1 ở giữa: lợi ích =', 3 + 1)

khối: [('1', 0, 0), ('0', 1, 3), ('1', 4, 4), ('0', 5, 5), ('1', 6, 6)] | giao dịch quanh khối 1 ở giữa: lợi ích = 4


### 3.3. Khối 0 bị cắt bởi truy vấn

Danh sách khối 0 của `s`: `(start_j, end_j)`. Truy vấn `[l, r]` gặp các khối có `end ≥ l` và `start ≤ r` — chỉ số liên tiếp `a..b` tìm bằng `bisect`. Độ dài trong đoạn: `min(end, r) − max(start, l) + 1` (chỉ khác độ dài thật ở `a` và `b`).

### 3.4. Sparse table cho max đoạn

In [4]:
def build_sparse(arr):
    sp = [arr[:]]
    k = 1
    while (1 << k) <= len(arr):
        prev, h = sp[-1], 1 << (k - 1)
        sp.append([max(prev[i], prev[i + h]) for i in range(len(arr) - (1 << k) + 1)])
        k += 1
    return sp

def query_max(sp, a, b):
    if a > b:
        return 0
    k = (b - a + 1).bit_length() - 1
    return max(sp[k][a], sp[k][b - (1 << k) + 1])

sp = build_sparse([3, 1, 4, 1, 5, 9, 2])
print(query_max(sp, 1, 3), query_max(sp, 2, 6))

4 9


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 3, truy vấn `[1, 5]`: đoạn `"00010"`, tăng cường `"1000101"`. Khối 1 duy nhất bị kẹp: `"1"` ở vị trí 4 của `s`, hai khối 0 bên cạnh dài `3` và `1` ⇒ lợi ích 4. Số `1` của `s` = 2 ⇒ `2 + 4 = 6` ✓.

Truy vấn `[0, 4]`: đoạn `"10001"`; khối 1 ở đầu và cuối chạm số `1` tăng cường ⇒ **không** bị kẹp giữa hai khối 0 ⇒ không giao dịch ⇒ `2`.

⇒ Lợi ích = max tổng **hai khối 0 liên tiếp** (giữa chúng luôn là một khối 1) trong đoạn, với độ dài tính **trong đoạn**. Cần ít nhất 2 khối 0 giao với đoạn.

In [5]:
s = "1000100"
zeros = [(a, b) for c, a, b in runs(s) if c == '0']
print('khối 0 của s:', zeros)
for l, r in [[1, 5], [0, 6], [0, 4]]:
    inside = [(max(a, l), min(b, r)) for a, b in zeros if b >= l and a <= r]
    lens = [y - x + 1 for x, y in inside]
    gain = max((lens[i] + lens[i + 1] for i in range(len(lens) - 1)), default=0)
    print([l, r], 'khối 0 trong đoạn:', inside, '-> lợi ích', gain, '-> đáp án', s.count('1') + gain)

khối 0 của s: [(1, 3), (5, 6)]
[1, 5] khối 0 trong đoạn: [(1, 3), (5, 5)] -> lợi ích 4 -> đáp án 6
[0, 6] khối 0 trong đoạn: [(1, 3), (5, 6)] -> lợi ích 5 -> đáp án 7
[0, 4] khối 0 trong đoạn: [(1, 3)] -> lợi ích 0 -> đáp án 2


## 5. Cách 1 — Brute Force: mô phỏng mọi giao dịch trên chuỗi tăng cường

In [6]:
def trade_brute_one(s, l, r):
    sub = s[l:r + 1]
    t = '1' + sub + '1'
    best = t.count('1') - 2
    bl = runs(t)
    for k, (c, a, b) in enumerate(bl):
        if c == '1' and 0 < k < len(bl) - 1:                  # khối 1 bị kẹp bởi 0
            u = t[:a] + '0' * (b - a + 1) + t[b + 1:]
            bl2 = runs(u)
            for k2, (c2, a2, b2) in enumerate(bl2):
                if c2 == '0' and 0 < k2 < len(bl2) - 1:      # khối 0 bị kẹp bởi 1
                    w = u[:a2] + '1' * (b2 - a2 + 1) + u[b2 + 1:]
                    best = max(best, w.count('1') - 2)
    return s.count('1') - sub.count('1') + best

def trade_brute(s, queries):
    return [trade_brute_one(s, l, r) for l, r in queries]

for s, q, exp in examples:
    print(trade_brute(s, q), '| kỳ vọng', exp)

[1] | kỳ vọng [1]
[4, 3, 1, 1] | kỳ vọng [4, 3, 1, 1]
[6, 7, 2] | kỳ vọng [6, 7, 2]
[4, 4, 2] | kỳ vọng [4, 4, 2]


**Độ phức tạp:** `O(L^2)` mỗi truy vấn (`L = r − l + 1`) ⇒ `O(q n^2)`.

**Nút thắt:** mô phỏng mọi lựa chọn; trong khi lợi ích chỉ là tổng hai khối 0 kề nhau.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề:** lợi ích tối ưu trong đoạn = `max_{a ≤ j < b} (len'(j) + len'(j+1))`, với `a..b` các khối 0 giao đoạn và `len'` độ dài trong đoạn (0 nếu `b − a < 1`).

Chỉ `len'(a)` và `len'(b)` có thể khác độ dài thật ⇒ tách:

- cặp `(a, a+1)` và `(b−1, b)` tính trực tiếp (nếu `b = a + 1` thì chỉ có một cặp, cả hai đều bị cắt);
- các cặp `(j, j+1)` với `a + 1 ≤ j ≤ b − 2` dùng **độ dài thật** ⇒ `pv[j] = len(j) + len(j+1)`, lấy max bằng sparse table.

Đáp án = `ones(s) + lợi ích`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Mỗi truy vấn quét các khối trong đoạn (`O(L)`)

In [7]:
def trade_scan(s, queries):
    total = s.count('1')
    zeros = [(a, b) for c, a, b in runs(s) if c == '0']
    starts = [a for a, _ in zeros]
    ends = [b for _, b in zeros]
    res = []
    for l, r in queries:
        a = bisect_left(ends, l)
        b = bisect_right(starts, r) - 1
        gain = 0
        prev = None
        for j in range(a, b + 1):
            ln = min(ends[j], r) - max(starts[j], l) + 1
            if prev is not None:
                gain = max(gain, prev + ln)
            prev = ln
        res.append(total + gain)
    return res

for s, q, exp in examples:
    print(trade_scan(s, q), '| kỳ vọng', exp)

[1] | kỳ vọng [1]
[4, 3, 1, 1] | kỳ vọng [4, 3, 1, 1]
[6, 7, 2] | kỳ vọng [6, 7, 2]
[4, 4, 2] | kỳ vọng [4, 4, 2]


**Độ phức tạp:** `O(log n + số khối trong đoạn)` mỗi truy vấn — xấu nhất `O(n)` (chuỗi `0101…`).

### 7.2. Cách 3 — Sparse table trên tổng cặp + xử lý biên (lời giải chính)

In [8]:
def trade_sparse(s, queries):
    total = s.count('1')
    zeros = [(a, b) for c, a, b in runs(s) if c == '0']
    starts = [a for a, _ in zeros]
    ends = [b for _, b in zeros]
    L = [b - a + 1 for a, b in zeros]
    pv = [L[j] + L[j + 1] for j in range(len(L) - 1)]
    sp = build_sparse(pv) if pv else [[]]
    res = []
    for l, r in queries:
        a = bisect_left(ends, l)
        b = bisect_right(starts, r) - 1
        gain = 0
        if a < b:
            la = min(ends[a], r) - max(starts[a], l) + 1
            lb = min(ends[b], r) - max(starts[b], l) + 1
            if b == a + 1:
                gain = la + lb
            else:
                gain = max(la + L[a + 1], L[b - 1] + lb, query_max(sp, a + 1, b - 2))
        res.append(total + gain)
    return res

for s, q, exp in examples:
    print(trade_sparse(s, q), '| kỳ vọng', exp)

[1] | kỳ vọng [1]
[4, 3, 1, 1] | kỳ vọng [4, 3, 1, 1]
[6, 7, 2] | kỳ vọng [6, 7, 2]
[4, 4, 2] | kỳ vọng [4, 4, 2]


**Độ phức tạp:** `O(n log n)` dựng + `O(log n)` mỗi truy vấn (do `bisect`; phần RMQ là `O(1)`).

### 7.3. Cách 4 — Cây phân đoạn thay sparse table

Nếu đề có **cập nhật** ký tự, sparse table không dùng được; cây phân đoạn max trên `pv` vẫn chạy. Ở đây minh họa phần truy vấn bằng cây phân đoạn lặp.

In [9]:
class MaxSeg:
    def __init__(self, arr):
        self.n = max(1, len(arr))
        self.t = [0] * (2 * self.n)
        for i, v in enumerate(arr):
            self.t[self.n + i] = v
        for i in range(self.n - 1, 0, -1):
            self.t[i] = max(self.t[2 * i], self.t[2 * i + 1])

    def query(self, l, r):          # max trên [l, r], 0 nếu rỗng
        res = 0
        l += self.n; r += self.n + 1
        while l < r:
            if l & 1: res = max(res, self.t[l]); l += 1
            if r & 1: r -= 1; res = max(res, self.t[r])
            l >>= 1; r >>= 1
        return res

def trade_segtree(s, queries):
    total = s.count('1')
    zeros = [(a, b) for c, a, b in runs(s) if c == '0']
    starts = [a for a, _ in zeros]; ends = [b for _, b in zeros]
    L = [b - a + 1 for a, b in zeros]
    seg = MaxSeg([L[j] + L[j + 1] for j in range(len(L) - 1)])
    res = []
    for l, r in queries:
        a = bisect_left(ends, l); b = bisect_right(starts, r) - 1
        gain = 0
        if a < b:
            la = min(ends[a], r) - max(starts[a], l) + 1
            lb = min(ends[b], r) - max(starts[b], l) + 1
            gain = la + lb if b == a + 1 else max(la + L[a + 1], L[b - 1] + lb, seg.query(a + 1, b - 2) if a + 1 <= b - 2 else 0)
        res.append(total + gain)
    return res

for s, q, exp in examples:
    print(trade_segtree(s, q), '| kỳ vọng', exp)

[1] | kỳ vọng [1]
[4, 3, 1, 1] | kỳ vọng [4, 3, 1, 1]
[6, 7, 2] | kỳ vọng [6, 7, 2]
[4, 4, 2] | kỳ vọng [4, 4, 2]


## 8. TRACE từng bước

Trace cách 3 với `s = "0010110001"` và vài truy vấn: in khối 0, chỉ số `a, b`, độ dài bị cắt và nguồn của lợi ích lớn nhất.

In [10]:
def trace(s, queries):
    zeros = [(a, b) for c, a, b in runs(s) if c == '0']
    L = [b - a + 1 for a, b in zeros]
    starts = [a for a, _ in zeros]; ends = [b for _, b in zeros]
    print('khối 0:', zeros, '| độ dài:', L, '| số 1 của s:', s.count('1'))
    for l, r in queries:
        a = bisect_left(ends, l); b = bisect_right(starts, r) - 1
        if a >= b:
            print(f"[{l},{r}] '{s[l:r+1]}': chỉ {max(0, b - a + 1)} khối 0 -> không giao dịch -> {s.count('1')}")
            continue
        la = min(ends[a], r) - max(starts[a], l) + 1
        lb = min(ends[b], r) - max(starts[b], l) + 1
        cands = {}
        if b == a + 1:
            cands['cặp (a, b) cắt cả hai'] = la + lb
        else:
            cands['cặp đầu (cắt)'] = la + L[a + 1]
            cands['cặp cuối (cắt)'] = L[b - 1] + lb
            if a + 1 <= b - 2:
                cands['các cặp bên trong'] = max(L[j] + L[j + 1] for j in range(a + 1, b - 1))
        g = max(cands.values())
        print(f"[{l},{r}] '{s[l:r+1]}': khối a={a}, b={b}, la={la}, lb={lb}, ứng viên {cands} -> {s.count('1') + g}")

S = "0010110001"
Q = [[0, 9], [1, 8], [2, 6], [3, 9], [4, 5]]
trace(S, Q)
print('vét cạn:', trade_brute(S, Q))

khối 0: [(0, 1), (3, 3), (6, 8)] | độ dài: [2, 1, 3] | số 1 của s: 4
[0,9] '0010110001': khối a=0, b=2, la=2, lb=3, ứng viên {'cặp đầu (cắt)': 3, 'cặp cuối (cắt)': 4} -> 8
[1,8] '01011000': khối a=0, b=2, la=1, lb=3, ứng viên {'cặp đầu (cắt)': 2, 'cặp cuối (cắt)': 4} -> 8
[2,6] '10110': khối a=1, b=2, la=1, lb=1, ứng viên {'cặp (a, b) cắt cả hai': 2} -> 6
[3,9] '0110001': khối a=1, b=2, la=1, lb=3, ứng viên {'cặp (a, b) cắt cả hai': 4} -> 8
[4,5] '11': chỉ 0 khối 0 -> không giao dịch -> 4
vét cạn: [8, 8, 6, 8, 4]


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def maxActiveSectionsAfterTrade(self, s: str, queries: List[List[int]]) -> List[int]:
        n = len(s)
        total = s.count('1')
        starts, ends = [], []
        i = 0
        while i < n:
            if s[i] == '0':
                j = i
                while j < n and s[j] == '0':
                    j += 1
                starts.append(i)
                ends.append(j - 1)
                i = j
            else:
                i += 1
        L = [e - b + 1 for b, e in zip(starts, ends)]
        pv = [L[j] + L[j + 1] for j in range(len(L) - 1)]
        sp = [pv]
        k = 1
        while (1 << k) <= len(pv):
            prev, h = sp[-1], 1 << (k - 1)
            sp.append([max(prev[x], prev[x + h]) for x in range(len(pv) - (1 << k) + 1)])
            k += 1

        res = []
        for l, r in queries:
            a = bisect_left(ends, l)
            b = bisect_right(starts, r) - 1
            gain = 0
            if a < b:
                la = min(ends[a], r) - max(starts[a], l) + 1
                lb = min(ends[b], r) - max(starts[b], l) + 1
                if b == a + 1:
                    gain = la + lb
                else:
                    gain = max(la + L[a + 1], L[b - 1] + lb)
                    if a + 1 <= b - 2:
                        x, y = a + 1, b - 2
                        kk = (y - x + 1).bit_length() - 1
                        gain = max(gain, sp[kk][x], sp[kk][y - (1 << kk) + 1])
            res.append(total + gain)
        return res

for s, q, exp in examples:
    got = Solution().maxActiveSectionsAfterTrade(s, q)
    print(got, 'OK' if got == exp else 'SAI')

[1] OK
[4, 3, 1, 1] OK
[6, 7, 2] OK
[4, 4, 2] OK


## 10. Vì sao thuật toán đúng

1. **Lợi ích một giao dịch** (mục 3.2): chọn khối 1 bị kẹp giữa hai khối 0 trong `t` cho lợi ích tối đa `Z_L + Z_R`, đạt được bằng cách đổi khối 0 vừa gộp. Không giao dịch thì lợi ích 0.
2. **Khối 1 bị kẹp trong `t`** ⇔ khối 1 nằm giữa hai khối 0 **của đoạn** (hai số 1 tăng cường làm các khối 1 chạm biên không bị kẹp). Hai khối 0 liên tiếp trong đoạn luôn có đúng một khối 1 ở giữa.
3. **Độ dài trong đoạn:** khối 0 giao đoạn ở biên bị cắt; các khối bên trong nguyên vẹn.
4. **Phân rã max:** mọi cặp liên tiếp `(j, j+1)` với `a ≤ j < b` hoặc là cặp đầu, cặp cuối, hoặc nằm hoàn toàn bên trong (dùng độ dài thật) ⇒ ba ứng viên phủ hết.
5. **Kết quả trên toàn `s`:** phần ngoài đoạn không đổi, nên đáp án = số 1 của `s` + lợi ích.

## 11. Phân tích Time / Space Complexity

Gọi `n = |s|`, `q` số truy vấn, `z` số khối 0.

| Cách | Mỗi truy vấn | Tiền xử lý | Space |
|---|---|---|---|
| 1. Mô phỏng | `O(L^2)` | — | `O(L)` |
| 2. Quét khối | `O(log n + số khối)` | `O(n)` | `O(z)` |
| 3. Sparse table | `O(log n)` | `O(z log z)` | `O(z log z)` |
| 4. Cây phân đoạn | `O(log n)` | `O(z)` | `O(z)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {'scan': trade_scan, 'sparse': trade_sparse, 'segtree': trade_segtree,
         'submit': lambda s, q: Solution().maxActiveSectionsAfterTrade(s, q)}
edge = [("1", [[0, 0]], [1]), ("0", [[0, 0]], [0]), ("000", [[0, 2]], [0]), ("111", [[0, 2]], [3]),
        ("010", [[0, 2]], [3]), ("101", [[0, 2]], [2])]
for s, q, exp in examples + edge:
    assert trade_brute(s, q) == exp, (s, q)
    for name, f in impls.items():
        got = f(s, q)
        assert got == exp, (name, s, q, got, exp)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với mô phỏng

In [13]:
import random
random.seed(3501)
cnt = 0
for _ in range(1500):
    n = random.randint(1, 12)
    s = ''.join(random.choice('01') for _ in range(n))
    qs = []
    for _ in range(6):
        l = random.randrange(n)
        qs.append([l, random.randrange(l, n)])
    exp = trade_brute(s, qs)
    for name, f in impls.items():
        assert f(s, qs) == exp, (name, s, qs)
    cnt += 1
print('Khớp mô phỏng trên', cnt, 'chuỗi ngẫu nhiên.')

Khớp mô phỏng trên 1500 chuỗi ngẫu nhiên.


### 12.3. Ca lớn (`n = q = 10^5`)

In [14]:
import time
random.seed(5)
n = 10**5
for name, s in [('ngẫu nhiên', ''.join(random.choice('01') for _ in range(n))), ('xen kẽ 01', '01' * (n // 2)),
                ('khối dài', ''.join(random.choice(['0' * 50, '1' * 30]) for _ in range(n // 40))[:n])]:
    qs = []
    for _ in range(10**5):
        l = random.randrange(len(s)); qs.append([l, random.randrange(l, len(s))])
    t = time.time(); a = Solution().maxActiveSectionsAfterTrade(s, qs); t1 = time.time() - t
    assert a[:300] == trade_scan(s, qs[:300])
    print(f"{name}: sparse table {t1:.2f}s")

ngẫu nhiên: sparse table 0.20s


xen kẽ 01: sparse table 0.25s


khối dài: sparse table 0.11s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm số `1` trong **đoạn** (quên phần còn lại của `s`)

In [15]:
def wrong_only_substring(s, queries):
    return [x - (s.count('1') - s[l:r + 1].count('1')) for x, (l, r) in zip(Solution().maxActiveSectionsAfterTrade(s, queries), queries)]

s, q, exp = examples[2]
print('sai :', wrong_only_substring(s, q), '| đúng:', exp)

sai : [5, 7, 2] | đúng: [6, 7, 2]


**Vì sao sai:** đề hỏi số đoạn hoạt động của **`s`** sau khi giao dịch trên đoạn con.

### Lỗi 2 — Dùng độ dài **thật** của khối 0 ở biên (không cắt theo truy vấn)

In [16]:
def wrong_no_truncate(s, queries):
    total = s.count('1')
    zeros = [(a, b) for c, a, b in runs(s) if c == '0']
    starts = [a for a, _ in zeros]; ends = [b for _, b in zeros]
    L = [b - a + 1 for a, b in zeros]
    res = []
    for l, r in queries:
        a = bisect_left(ends, l); b = bisect_right(starts, r) - 1
        gain = max((L[j] + L[j + 1] for j in range(a, b)), default=0)     # SAI
        res.append(total + gain)
    return res

s, q, exp = examples[1]
print('sai :', wrong_no_truncate(s, q), '| đúng:', exp)

sai : [4, 4, 1, 1] | đúng: [4, 3, 1, 1]


**Vì sao sai:** truy vấn `[0, 2]` của `"0100"` chỉ thấy **một** số 0 ở khối cuối, không phải hai.

### Lỗi 3 — Coi khối 1 chạm biên đoạn là "bị kẹp" (bỏ qua số 1 tăng cường)

In [17]:
def wrong_edges(s, queries):
    total = s.count('1')
    out = []
    for l, r in queries:
        sub = s[l:r + 1]
        bl = runs(sub)
        gain = 0
        for k, (c, a, b) in enumerate(bl):
            if c == '1':                                         # SAI: không kiểm tra có 0 cả hai bên
                left = bl[k - 1][2] - bl[k - 1][1] + 1 if k > 0 else 0
                right = bl[k + 1][2] - bl[k + 1][1] + 1 if k + 1 < len(bl) else 0
                gain = max(gain, left + right)
        out.append(total + gain)
    return out

s, q, exp = examples[2]
print('sai :', wrong_edges(s, q), '| đúng:', exp)

sai : [6, 7, 5] | đúng: [6, 7, 2]


**Vì sao sai:** đoạn `"10001"` (truy vấn `[0,4]`): hai khối 1 chạm biên, bên ngoài là số 1 tăng cường ⇒ không bị kẹp ⇒ không giao dịch được.

### Lỗi 4 — Truy vấn sparse table khi đoạn bên trong **rỗng**

In [18]:
sp_demo = build_sparse([5, 7])
try:
    a, b = 1, 0                             # a + 1 > b - 2 ... đoạn rỗng
    k = (b - a + 1).bit_length() - 1        # (0).bit_length() - 1 = -1
    print(sp_demo[k][a])                    # chỉ số -1 -> lấy nhầm tầng cuối!
except Exception as e:
    print('lỗi:', e)
print('cách đúng: kiểm tra a <= b trước (query_max trả 0 cho đoạn rỗng):', query_max(sp_demo, 1, 0))

lỗi: list index out of range
cách đúng: kiểm tra a <= b trước (query_max trả 0 cho đoạn rỗng): 0


**Vì sao sai:** với đoạn rỗng, `bit_length() − 1 = −1` nên Python lấy `sp[−1]` (tầng cuối của bảng) — có lúc báo lỗi chỉ số như trên, có lúc **lặng lẽ trả giá trị sai**. Luôn kiểm tra `a + 1 ≤ b − 2`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Chuỗi nhị phân + thao tác trên "khối" ⇒ **nén run-length**, làm việc trên danh sách khối.
- Truy vấn đoạn cắt khối ở hai biên ⇒ xử lý **riêng hai khối biên**, phần giữa dùng cấu trúc tĩnh (sparse table / cây phân đoạn).
- "Đổi 1→0 rồi 0→1" ⇒ tính **lợi ích ròng**, thường đơn giản hơn mô phỏng.
- Tìm khối giao đoạn ⇒ `bisect` trên mảng đầu/cuối khối.

## 15. Biến thể của bài

1. Bản I (LC 3499): chỉ một truy vấn trên cả chuỗi ⇒ quét `O(n)`.
2. Cho phép **k** giao dịch ⇒ phức tạp hơn (các giao dịch tương tác).
3. Có **cập nhật** ký tự ⇒ danh sách khối thay đổi; dùng cây phân đoạn lưu thông tin khối ở biên mỗi nút.
4. Bài liên quan: LC 487 *Max Consecutive Ones II*, LC 1004.

In [19]:
# Biến thể 1: một truy vấn trên cả chuỗi
def trade_whole(s):
    return Solution().maxActiveSectionsAfterTrade(s, [[0, len(s) - 1]])[0]

print(trade_whole("0100"), trade_whole("1000100"), trade_whole("01"))

4 7 1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Mỗi truy vấn | Khi nào dùng |
|---|---|---|---|
| 1 | Mô phỏng | `O(L^2)` | Oracle |
| 2 | Quét khối | `O(số khối)` | Ít khối |
| 3 | Sparse table | `O(log n)` | **Nộp bài** |
| 4 | Cây phân đoạn | `O(log n)` | Có cập nhật |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `s = "00100"`, truy vấn `[[0, 4], [1, 3], [2, 4]]`.

**Bài 2 (tính tay):** `s = "0011000111"`, truy vấn `[0, 9]`.

**Bài 3 (điền TODO):** viết `zero_blocks(s)` trả về danh sách `(start, end)`.

**Bài 4 (tự viết lại):** trả về cả **khối 1** được chọn cho giao dịch tối ưu (vị trí trong `s`) hoặc `None`.

**Bài 5 (mini challenge):** biến thể "giao dịch ngược": đổi khối **0** kẹp giữa **1** thành 1 trước rồi đổi một khối 1 kẹp giữa 0 thành 0 — tối đa hóa số 1. Dùng vét cạn để kiểm tra một công thức bạn đề xuất.

In [20]:
def zero_blocks_todo(s):
    # TODO
    raise NotImplementedError

def chosen_block_todo(s, l, r):
    # TODO
    raise NotImplementedError

def reverse_trade_todo(s, l, r):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Khối 0: `[0,1]`, `[3,4]`; số 1 = 1. (2) `[0,4]`: `2+2 = 4` ⇒ 5; `[1,3]`: `1+1 = 2` ⇒ 3; `[2,4]`: chỉ một khối 0 ⇒ 1. (3) `[5, 3, 1]`.

**Bài 2.** (1) Khối 0 độ dài 2 và 3. (2) Lợi ích 5. (3) `5 + 5 = 10`.

**Bài 3.** (1) Duyệt, ghi đầu/cuối mỗi đoạn `0`. (2) So với `runs`. (3) Rỗng nếu không có `0`.

**Bài 4.** (1) Giữ chỉ số `j` đạt max. (2) Khối 1 nằm giữa `end_j + 1` và `start_{j+1} − 1`. (3) Kiểm tra bằng mô phỏng.

**Bài 5.** (1) Thử vét cạn trên nhiều chuỗi nhỏ. (2) Quan sát: bước 1 lấp một khối 0 `Z` (được `+Z`), bước 2 phải bỏ một khối 1 kẹp giữa 0 (mất `−O`, có thể chọn khối nhỏ nhất). (3) So sánh với "không giao dịch".

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
a1 = Solution().maxActiveSectionsAfterTrade("00100", [[0, 4], [1, 3], [2, 4]])
a2 = Solution().maxActiveSectionsAfterTrade("0011000111", [[0, 9]])
assert a1 == [5, 3, 1] == trade_brute("00100", [[0, 4], [1, 3], [2, 4]])
assert a2 == [10] == trade_brute("0011000111", [[0, 9]])

def zero_blocks(s):
    return [(a, b) for c, a, b in runs(s) if c == '0']

assert zero_blocks("1000100") == [(1, 3), (5, 6)] and zero_blocks("111") == []
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3:', zero_blocks("0011000111"))

Bài 1: [5, 3, 1] | Bài 2: [10] | Bài 3: [(0, 1), (4, 6)]


In [22]:
# Bài 4: khối 1 được chọn
def chosen_block(s, l, r):
    zeros = zero_blocks(s)
    inside = [(max(a, l), min(b, r)) for a, b in zeros if b >= l and a <= r]
    best, pick = 0, None
    for j in range(len(inside) - 1):
        g = (inside[j][1] - inside[j][0] + 1) + (inside[j + 1][1] - inside[j + 1][0] + 1)
        if g > best:
            best, pick = g, (inside[j][1] + 1, inside[j + 1][0] - 1)
    return pick, s.count('1') + best

random.seed(4)
for _ in range(500):
    n = random.randint(1, 12)
    s = ''.join(random.choice('01') for _ in range(n))
    l = random.randrange(n); r = random.randrange(l, n)
    pick, val = chosen_block(s, l, r)
    assert val == trade_brute_one(s, l, r)
    if pick:
        a, b = pick
        assert set(s[a:b + 1]) == {'1'}
print('Bài 4: ví dụ', chosen_block("1000100", 1, 5))

Bài 4: ví dụ ((4, 4), 6)


In [23]:
# Bài 5: giao dịch ngược — thử nghiệm bằng vét cạn và một công thức đề xuất
def reverse_trade_brute(s, l, r):
    sub = s[l:r + 1]
    t = '1' + sub + '1'
    best = t.count('1') - 2
    bl = runs(t)
    for k, (c, a, b) in enumerate(bl):
        if c == '0' and 0 < k < len(bl) - 1:
            u = t[:a] + '1' * (b - a + 1) + t[b + 1:]
            bl2 = runs(u)
            for k2, (c2, a2, b2) in enumerate(bl2):
                if c2 == '1' and 0 < k2 < len(bl2) - 1:
                    w = u[:a2] + '0' * (b2 - a2 + 1) + u[b2 + 1:]
                    best = max(best, w.count('1') - 2)
    return s.count('1') - sub.count('1') + best

def reverse_trade_formula(s, l, r):
    # đề xuất: giao dịch ngược không bao giờ có lợi hơn "không làm gì"?
    return s.count('1')

random.seed(5)
mism = sum(reverse_trade_brute(s, l, r) != reverse_trade_formula(s, l, r)
           for s, l, r in ((lambda s: (s, 0, len(s) - 1))(''.join(random.choice('01') for _ in range(random.randint(1, 10)))) for _ in range(500)))
print('số ca công thức "không làm gì" sai:', mism)

số ca công thức "không làm gì" sai: 43


Kết quả cho thấy giả thuyết "giao dịch ngược vô ích" là **sai** (có hàng chục ca giao dịch ngược vẫn có lợi) — đây là cách **dùng vét cạn để bác bỏ giả thuyết** trước khi mất công chứng minh; bạn đọc có thể tiếp tục tìm công thức đúng. (Lấp khối 0 được `+Z`, nhưng bước 2 bắt buộc xóa một khối 1 bị kẹp giữa hai khối 0 — sau bước 1 có thể không còn khối nào như vậy, khi đó không thực hiện được giao dịch.)

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
khối 0 của s: starts[], ends[], L[]; pv[j] = L[j] + L[j+1]; sparse table trên pv
truy vấn [l, r]: a = bisect_left(ends, l); b = bisect_right(starts, r) - 1
    a >= b -> gain 0
    la, lb = độ dài khối a, b bị cắt bởi [l, r]
    b == a+1 -> gain = la + lb
    else gain = max(la + L[a+1], L[b-1] + lb, RMQ(pv, a+1, b-2))
đáp án = ones(s) + gain
```

**Checklist:**

- [ ] Cộng số `1` của **toàn chuỗi** `s`?
- [ ] Cắt độ dài hai khối 0 ở biên?
- [ ] Khối 1 chạm biên đoạn không được chọn?
- [ ] RMQ chỉ khi đoạn bên trong khác rỗng?
- [ ] Đối chiếu với mô phỏng trên chuỗi tăng cường?